In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
from rdkit.Chem import MolFromSmiles
from rdkit.Chem.Draw import MolsToGridImage

import sys
sys.path.append('../')
import FragShapley

/home/jannik/Documents/studies/phd/03_work/20_FragShapley/FragShapley/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
plt.style.use('style.mplstyle')
sns.set_context('talk', font_scale=1.0)
fig_folder = os.path.abspath("figures/")

In [3]:
models = ['RF', 'GCN']#, 'MPNN']

In [4]:
# only select a single dataset
dataset_of_interest = 'Hansen'
# consider dummy atoms
remove_dummies = False

files = [f'../4_mutagenicity/{model.lower()}_classification_hansen/df_explanation.pkl' for model in models]
df_mut = pd.concat((pd.read_pickle(mut_file) for mut_file in files))

df_mut = df_mut.loc[df_mut.dataset == dataset_of_interest]

In [5]:
df_mut['fragments'] = df_mut.smiles.apply(lambda x: FragShapley.utils.get_BRICS_fragments_as_SMILES(x, remove_dummies=remove_dummies))

In [6]:
# get the values of the Explainer as a list (currently only available as dict)
df_mut['fragExplainer_shapley_values'] = df_mut.fragExplainer_result.apply(lambda x: list(x.values()))
# create a dataframe of all fragments (contains duplicates) and the corresponding Shapley Value
df_fragments = df_mut[['fragments', 'model', 'fragExplainer_shapley_values']].explode(['fragments', 'fragExplainer_shapley_values'], ignore_index=True)

In [7]:
df_fragments = df_fragments.groupby(['fragments', 'model']).agg([len, 'mean', 'std', list]) # will throw warning veacuse of error when calculating std for a single measurement
df_fragments = df_fragments.reset_index() # reset so that 'fragments' is a column again and no longer the index
df_fragments.columns = [col[0] if col[1]=='' else col[1] for col in df_fragments.columns.values] # rename, choose 'fragments' for first column, for the other simply len, mean, std
df_fragments

,fragments,model,len,mean,std,list
0,*=C(Br)C(=O)O,GCN,1,3.168824,NaN,[3.168823719024658]
1,*=C(Br)C(=O)O,RF,1,1.096623,NaN,[1.096622594085369]
2,*=C(Br)C=O,GCN,2,4.67662,0.626745,"[4.233443737030029, 5.1197956800460815]"
3,*=C(Br)C=O,RF,2,3.541465,0.866576,"[2.928703233823888, 4.154227005869744]"
4,*=C(C#N)C#N,GCN,5,-4.622884,4.056222,"[-6.165920197963715, 2.6152063409487405, -6.32..."
...,...,...,...,...,...,...
10359,c1cscn1,RF,1,1.558517,NaN,[1.5585174312995131]
10360,c1scc2c1-c1cscc1C1NC21,GCN,1,6.313836,NaN,[6.313836097717285]
10361,c1scc2c1-c1cscc1C1NC21,RF,1,2.750783,NaN,[2.7507829670476576]
10362,c1scc2c1-c1cscc1C1OC21,GCN,1,2.831149,NaN,[2.8311493396759033]


In [15]:
from rdkit.Chem import Draw

do = Draw.MolDrawOptions()
do.legendFontSize = 32

min_n = 20
mols_per_row = 5

top_k = 2 * mols_per_row

all_smiles = []
all_avg_contributions = []

for model in models:
    df_tmp = df_fragments.loc[df_fragments.model == model]
    df_tmp = df_tmp.sort_values(by='mean', ascending=False)
    df_tmp = df_tmp.loc[df_tmp.len >= min_n]
    smiles = df_tmp.iloc[:top_k].fragments.to_list()
    avg_contribution = df_tmp.iloc[:top_k]['mean'].to_list()
    all_smiles += smiles
    all_avg_contributions += avg_contribution
    print(f'{model}:')
    print('\t', smiles)
    print('\t', avg_contribution)

mols = [MolFromSmiles(sm) for sm in all_smiles]
svg = MolsToGridImage(mols=mols,
                      molsPerRow=mols_per_row,
                      legends=[f'{c:.2f}' for c in all_avg_contributions],
                      subImgSize=(200, 200),
                      useSVG=True,
                      drawOptions=do)
with open(os.path.join(fig_folder, "2_mutagenicity_fragments.svg"), "w") as f:
    f.write(svg.data)

RF:
	 ['*=Cc1ccc([N+](=O)[O-])o1', '*C1CO1', '*c1csc([N+](=O)[O-])c1', '*[C@H]1CO1', '*c1ccc([N+](=O)[O-])cc1', '*ON(O*)C(*)=O', '*c1ccc(N)cc1', '*c1ccccc1[N+](=O)[O-]', '*=CC=O', '*C(=O)Cl']
	 [7.318923903265354, 5.3690050732917225, 4.623572031163877, 4.311224986607315, 3.580493972304765, 2.7077133772275643, 2.67408606986163, 2.5170614437778487, 2.401859115710965, 2.053538633075832]
GCN:
	 ['*c1csc([N+](=O)[O-])c1', '*[C@H]1CO1', '*=Cc1ccc([N+](=O)[O-])o1', '*C1CO1', '*c1ccc(N)cc1', '*c1ccc(N)c(*)c1', '*c1ccc([N+](=O)[O-])cc1', '*CCl', '*CBr', '*c1ccccc1N']
	 [10.579115865503748, 9.22935162418719, 7.957726960783607, 2.518633434025273, 2.4353188107679493, 2.386332733483046, 2.3855471378499096, 2.1268021991766157, 1.9919634103775024, 1.7483419802152746]


In [9]:
# only correctly predicted and mutagenic compounds
df_plot = df_mut.query('y_true == y_pred')
df_plot = df_plot.query('y_true == 1')

# get number of fragments, more interesting to plot compounds with more than two fragments
df_plot['n_fragments'] = df_plot.fragments.apply(len)
df_plot = df_plot.loc[df_plot.n_fragments > 2]

In [11]:
model = 'RF'
row_indices = {'RF': [3, 7, 9, 13, 20, 44]} # 3 (epoxide), 7 (nitro), 8 (azo), 9 (aromatic amine), 13 (N(H)OH), 20 (PAH)

df_tmp = df_plot.loc[df_plot.model == model]
for idx in row_indices[model]:
    row = df_tmp.iloc[idx]
    contributions = FragShapley.visualization.get_atom_contribution_from_result_dict(row.smiles,
                                                                                     results_dict=row.fragExplainer_result,
                                                                                     frag_to_atom_ids=row.frag_to_atom_ids)
    out = FragShapley.visualization.visualize_contributions(smiles=row.smiles,
                                                            contributions=contributions,
                                                            scale=0.3)
    with open(os.path.join(fig_folder, f"molecules_mutagenicity/2_mutagenicity_ex_mol_{model}_idx_{idx}.svg"), "w") as f:
        f.write(out.data)